In [1]:
import sys
import os
import matplotlib.pyplot as plt
import cv2
import numpy as np

# Add the src directory to the path. TEMPORARY FIX
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../../..")))

from src.data_processing.dataset_loader import CoastData

In [3]:
data_path = os.path.abspath(os.path.join(os.getcwd(), "../../../data/processed_obliques/"))

# Load the data, all the different stations
data = CoastData(data_path)
station_names = data.get_station_names()

CLASS_MAP = {
    0: "NoData",
    1: "Landwards",
    2: "Seawards"
}

total_pixels_station_and_class = {station: {cls: 0 for cls in CLASS_MAP.keys()} for station in station_names} 
total_pixels_station = {station: 0 for station in station_names}
total_pixels_global = {cls: 0 for cls in CLASS_MAP.keys()}
print(total_pixels_station_and_class)

station_shapes = {station: {'heights': [], 'widths': []} for station in station_names}

CoastData: global - 1717 images
{'agrelo': {0: 0, 1: 0, 2: 0}, 'arenaldentem': {0: 0, 1: 0, 2: 0}, 'cadiz': {0: 0, 1: 0, 2: 0}, 'cies': {0: 0, 1: 0, 2: 0}, 'samarador': {0: 0, 1: 0, 2: 0}}


In [4]:
for station in station_names:
    # print(station)
    station_data = data.get_images_and_masks(station)
    print(f"\nStation: {station}")
    print(f"Number of images in station {station}: {len(station_data)}")

    for image_data in station_data:
        mask = cv2.imread(image_data['mask'], cv2.IMREAD_GRAYSCALE)
        total_pixels = mask.shape[0] * mask.shape[1]
        total_pixels_station[station] += total_pixels

        classes, count = np.unique(mask, return_counts=True)
        for cls, cnt in zip(classes, count):
            percentage = (cnt / total_pixels) * 100
            total_pixels_station_and_class[station][cls] += cnt
            total_pixels_global[cls] += cnt

        
        height, width = mask.shape
        station_shapes[station]['heights'].append(height)
        station_shapes[station]['widths'].append(width)

    for cls, cnt in total_pixels_station_and_class[station].items():
        percentage = (cnt / total_pixels_station[station]) * 100 if total_pixels_station[station] > 0 else 0
        print(f"Class {CLASS_MAP[cls]}: {cnt} pixels ({percentage:.2f}%)")

    heights = station_shapes[station]['heights']
    widths = station_shapes[station]['widths']
    print(f"Image heights in station {station}: min={min(heights)}, max={max(heights)}, mean={np.mean(heights):.2f}, std={np.std(heights):.2f}")
    print(f"Image widths in station {station}: min={min(widths)}, max={max(widths)}, mean={np.mean(widths):.2f}, std={np.std(widths):.2f}")

print("\nGlobal statistics:")
for cls, cnt in total_pixels_global.items():
    total_pixels = sum(total_pixels_global.values())
    percentage = (cnt / total_pixels) * 100 if total_pixels > 0 else 0
    print(f"Class {CLASS_MAP[cls]}: {cnt} pixels ({percentage:.2f}%)")



Station: agrelo
Number of images in station agrelo: 244
Class NoData: 2067570 pixels (0.16%)
Class Landwards: 714391602 pixels (54.56%)
Class Seawards: 593004472 pixels (45.29%)
Image heights in station agrelo: min=488, max=2583, mean=670.84, std=175.83
Image widths in station agrelo: min=4726, max=8320, mean=7992.37, std=550.48

Station: arenaldentem
Number of images in station arenaldentem: 40
Class NoData: 707566 pixels (1.00%)
Class Landwards: 19439273 pixels (27.58%)
Class Seawards: 50339330 pixels (71.42%)
Image heights in station arenaldentem: min=267, max=1001, mean=453.98, std=146.49
Image widths in station arenaldentem: min=3624, max=4000, mean=3871.82, std=114.36

Station: cadiz
Number of images in station cadiz: 946
Class NoData: 1959448 pixels (0.48%)
Class Landwards: 221552983 pixels (53.90%)
Class Seawards: 187516087 pixels (45.62%)
Image heights in station cadiz: min=93, max=633, mean=236.27, std=97.08
Image widths in station cadiz: min=1263, max=2048, mean=1833.60, st

In [ ]:
station_shapes = {station: {'heights': [], 'widths': []} for station in station_names}
